### Unsupervised Learning — Earthquake Clustering

This notebook applies K-Means clustering on scaled seismic features to discover natural earthquake patterns (seismic profiles), then analyzes each cluster's felt rate to complement the supervised FeltQuake model.

In [21]:
import pandas as pd 
import plotly.express as px

In [22]:
df = pd.read_csv('earthquakes_cleaned.csv')
df

,mag,status,tsunami,sig,net,nst,dmin,rms,gap,magType,type,longitude,latitude,depth,reported,hour,month,day_of_week,mag_depth_ratio,is_remote
0,2.70,reviewed,0,112,ak,36.0,0.30000,0.90,146.0,ml,earthquake,-141.4430,60.0150,13.100,0,23,12,6,0.191489,False
1,4.30,reviewed,0,284,us,34.0,2.00700,0.29,188.0,mb,earthquake,145.2408,43.3059,51.870,0,23,12,6,0.081332,False
2,3.08,reviewed,0,175,nc,146.0,0.03365,0.12,16.0,mw,earthquake,-122.6045,38.3775,4.640,1,23,12,6,0.546099,False
3,4.00,reviewed,0,246,us,18.0,2.42300,0.96,93.0,mb,earthquake,110.4116,-9.4878,10.000,0,23,12,6,0.363636,False
4,4.30,reviewed,0,284,us,40.0,1.71400,0.76,62.0,mb,earthquake,-66.7998,-24.1137,198.342,0,23,12,6,0.021571,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38265,2.90,reviewed,0,129,ak,96.0,0.50000,0.90,22.0,ml,earthquake,-148.8040,62.3460,26.900,0,0,12,1,0.103943,False
38266,4.20,reviewed,0,271,us,19.0,2.01200,0.82,83.0,mb,earthquake,168.4113,-17.1006,215.635,0,0,12,1,0.019387,False
38267,2.50,reviewed,0,96,ak,18.0,0.40000,0.90,81.0,ml,earthquake,-139.8570,60.4480,1.700,0,0,12,1,0.925926,False
38268,3.10,reviewed,0,148,us,34.0,0.25900,0.34,199.0,ml,earthquake,-166.3976,53.5945,65.389,0,0,12,1,0.046694,False


In [23]:
cluster_features = df[['mag', 'longitude' , 'latitude' , 'depth' , 'mag_depth_ratio' , 'is_remote' ]]
cluster_features

,mag,longitude,latitude,depth,mag_depth_ratio,is_remote
0,2.70,-141.4430,60.0150,13.100,0.191489,False
1,4.30,145.2408,43.3059,51.870,0.081332,False
2,3.08,-122.6045,38.3775,4.640,0.546099,False
3,4.00,110.4116,-9.4878,10.000,0.363636,False
4,4.30,-66.7998,-24.1137,198.342,0.021571,False
...,...,...,...,...,...,...
38265,2.90,-148.8040,62.3460,26.900,0.103943,False
38266,4.20,168.4113,-17.1006,215.635,0.019387,False
38267,2.50,-139.8570,60.4480,1.700,0.925926,False
38268,3.10,-166.3976,53.5945,65.389,0.046694,False


In [24]:
cluster_features['is_remote'] = cluster_features['is_remote'].astype('int')
cluster_features['is_remote'] 

0        0
1        0
2        0
3        0
4        0
        ..
38265    0
38266    0
38267    0
38268    0
38269    0
Name: is_remote, Length: 38270, dtype: int64

### Numerical Pipeline

In [25]:
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

scaler = RobustScaler()

num_pipeline = Pipeline(steps= [('Scaling' , scaler)])

num_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False


In [26]:
from sklearn import preprocessing
from sklearn.compose import ColumnTransformer

preprocessing_pipeline = ColumnTransformer(
                         
                         transformers= [('Numerical Pipeline', num_pipeline, cluster_features.columns)]
                      
)
preprocessing_pipeline

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical Pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name`

In [27]:
scaled_features = preprocessing_pipeline.fit_transform(cluster_features)
scaled_features

array([[-1.        , -0.27267292,  0.50701352, -0.0942093 ,  0.02463249,
         0.        ],
       [ 0.06666667,  0.74823352,  0.16997138,  0.8074186 , -0.31153627,
         0.        ],
       [-0.74666667, -0.20558735,  0.07055977, -0.29095349,  1.10679632,
         0.        ],
       ...,
       [-1.13333333, -0.26702503,  0.51574763, -0.35932558,  2.26591403,
         0.        ],
       [-0.73333333, -0.36153846,  0.3775045 ,  1.12181395, -0.41723834,
         0.        ],
       [-0.06666667, -0.04586967, -0.73488322,  3.82053488, -0.4911524 ,
         0.        ]], shape=(38270, 6))

### Use Elbow Method & Silhouette Score Method to find the best (K) value.


- Elbow Method

In [28]:
from sklearn.cluster import KMeans

k_values = range(2 , 11)

inertia_list = []

for k in k_values:

    kmeans = KMeans(n_clusters= k , random_state= 42)
    kmeans.fit(scaled_features)

    inertia_list.append(round(kmeans.inertia_, 2))

print(inertia_list)



[295673.31, 115508.0, 100276.09, 73060.56, 65085.88, 52105.75, 46269.65, 41699.57, 36994.45]


In [29]:
px.line(x= k_values, y= inertia_list, markers= True,
        labels= {'x' : 'Num of Clusters', 'y' : 'Inertia Score'})

- Silhouette Score Method

`sample_size` --> With ~38K rows, silhouette_score computes pairwise distances between all points, which scales as O(n²). On the full dataset, this can be very slow or even run out of memory. Using sample_size=5000 computes the score on a random subset instead, which is fast and still gives a reliable estimate. random_state=42 inside silhouette_score makes the random sample fixed, so results are reproducible across runs.

In [30]:
from sklearn.metrics import silhouette_score

k_values = range(2, 11)
silhouette_list = []

for k in k_values:

    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(scaled_features)
    silhouette_list.append(round(
        silhouette_score(scaled_features, labels, sample_size=5000, random_state=42), 2))

In [31]:
silhouette_list

[0.68, 0.45, 0.32, 0.34, 0.34, 0.34, 0.35, 0.35, 0.36]

In [32]:
px.line(x= k_values, y= silhouette_list,
        markers= True,
        labels= {'x' : 'Num of Clusters', 'y' : 'Silouhette Score'})

**Choosing K = 3**

- K=3 was chosen over K=2 primarily for interpretability: it separates events into three distinct seismic profiles (shallow/high-impact, intermediate, deep/remote/low-impact) that align meaningfully with the felt-rate analysis, rather than the coarser 2-cluster split.

- Elbow also shows the sharpest inertia drop up to k=3, then flattens.


In [33]:
best_k = 3

final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = final_kmeans.fit_predict(scaled_features)
df['cluster'] 

0        0
1        0
2        0
3        0
4        1
        ..
38265    0
38266    1
38267    0
38268    0
38269    1
Name: cluster, Length: 38270, dtype: int32

In [34]:
df['cluster'].value_counts()

cluster
0    30824
1     6052
2     1394
Name: count, dtype: int64

In [40]:
cluster_summary = df.groupby('cluster').agg(
    events=('reported', 'size'),
    felt_rate=('reported', 'mean'),
    avg_mag=('mag', 'mean'),
    avg_depth=('depth', 'mean'),
    avg_ratio=('mag_depth_ratio', 'mean'),
    remote_pct=('is_remote', 'mean')
).round(3).reset_index()

cluster_summary['felt_rate'] = (cluster_summary['felt_rate'] * 100).round(2)
cluster_summary['remote_pct'] = (cluster_summary['remote_pct'] * 100).round(2)

cluster_summary

,cluster,events,felt_rate,avg_mag,avg_depth,avg_ratio,remote_pct
0,0,30824,13.1,3.838,20.307,0.322,10.0
1,1,6052,10.1,4.138,131.941,0.010,10.6
2,2,1394,1.2,4.365,518.807,0.009,52.4


In [66]:
fig = px.bar(
    cluster_summary,
    x='cluster', y='felt_rate',
    color='felt_rate',
    color_continuous_scale=['#3B5E55', '#3B5E49'],
    title='Felt Rate per Cluster',
    text='felt_rate'
)
fig.update_layout(xaxis_title='Cluster', yaxis_title='Felt Rate (%)')
fig.show()

In [ ]:
# Saved data with clusters

# df.to_csv('earthquakes_clusters.csv')

In [69]:
cluster_names = {
    0: "Shallow & Near (High Impact)",
    1: "Intermediate Depth (Moderate)",
    2: "Deep & Remote (Low Impact)"
}

df['cluster_name'] = df['cluster'].map(cluster_names)

cluster_summary = df.groupby('cluster_name').agg(
    events=('reported', 'size'),
    felt_rate=('reported', 'mean')
).round(3)

cluster_summary['felt_rate'] = (cluster_summary['felt_rate'] * 100).round(2)
cluster_summary

,events,felt_rate
cluster_name,,
Deep & Remote (Low Impact),1394,1.2
Intermediate Depth (Moderate),6052,10.1
Shallow & Near (High Impact),30824,13.1


In [108]:
px.scatter(df, x='mag', y='depth', color='cluster_name',
          
        color_discrete_map={
        "Shallow & Near (High Impact)": "#3B5E64",
        "Intermediate Depth (Moderate)": "#b45356",
        "Deep & Remote (Low Impact)": "#505797"
          }
        , title='Clusters: Magnitude vs Depth',
          labels={
            'mag' : 'Magnitude', 
            'depth' : 'Depth' , 
            'cluster_name' : 'Cluster Name'
           } , 
          opacity=0.6)

In [71]:
df.head()

,mag,status,tsunami,sig,net,nst,dmin,rms,gap,magType,...,latitude,depth,reported,hour,month,day_of_week,mag_depth_ratio,is_remote,cluster,cluster_name
0,2.70,reviewed,0,112,ak,36.0,0.30000,0.90,146.0,ml,...,60.0150,13.100,0,23,12,6,0.191489,False,0,Shallow & Near (High Impact)
1,4.30,reviewed,0,284,us,34.0,2.00700,0.29,188.0,mb,...,43.3059,51.870,0,23,12,6,0.081332,False,0,Shallow & Near (High Impact)
2,3.08,reviewed,0,175,nc,146.0,0.03365,0.12,16.0,mw,...,38.3775,4.640,1,23,12,6,0.546099,False,0,Shallow & Near (High Impact)
3,4.00,reviewed,0,246,us,18.0,2.42300,0.96,93.0,mb,...,-9.4878,10.000,0,23,12,6,0.363636,False,0,Shallow & Near (High Impact)
4,4.30,reviewed,0,284,us,40.0,1.71400,0.76,62.0,mb,...,-24.1137,198.342,0,23,12,6,0.021571,False,1,Intermediate Depth (Moderate)


In [88]:
# Save Model

import joblib 

joblib.dump({'scaler': preprocessing_pipeline, 'model': final_kmeans}, 'Unsupervised_model.pkl')

['Unsupervised_model.pkl']